# Plot regional windowed TCC skill

Plots the cached regional TCC/RMSE ensemble summaries across variables and experiments.


## 1. Libraries


In [ ]:
from pathlib import Path
import sys


## 2. User setup


In [ ]:
from pathlib import Path
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")

groups = ("Jan2012", "Jun2012")
variables = ("UBOT", "VBOT", "U850", "V850", "T850", "Q850", "TREFHT", "PRECT", "FLUT")
region = "GLOBAL"
metric = "TCC_regional_ensmean"
vmin, vmax, nlev = 0.0, 1.0, 11
show = True


window_scheme = "biweekly"  # "biweekly", "weekly", "15day", or "month"


## Notes: available selections

- Experiment groups: Dec2011 (analysis), Jan2012, and Jun2012 (forecasts).
- Runs: da for Dec2011; fc (standard forecast) or wc (water-cycle forecast) for forecast groups.
- Observations and variables: `ERA5` supports UBOT, VBOT, PRECT, PSL, T925, Q925, U850, V850, T850, Q850, Z500, OMEGA850/500/200, U/V500, U/V200, FLDS/C, FLNS/C, FSDS/C, FSNS/C, LHFLX, SHFLX, QFLX, TREFHT, TS, PRECSL, PRECC, and PRECL; `GPCP` and `GPM` support PRECT; `NOAA-OLR` supports FLUT.
- Regions: GLOBAL, land/ocean where applicable, NINO3.4, MAR_CONT, CONUS, NH_POLAR.
- Existing NetCDF diagnostics are reused. Set `force_compute = True` to rebuild selected products.

Common metrics: `TCC_regional_ensmean`, `RMSE_regional_ensmean`, `TCC_member_ens_q50`, and `TCC_member_qpre90`; inspect a cached dataset for all generated quantile names.

Exact base-map fields are `TCC`, `RMSE`, and `BIAS` for members and `TCC_ensmean`, `RMSE_ensmean`, and `BIAS_ensmean` for the ensemble mean. For each base metric, regional outputs provide `<METRIC>_regional_member`, `<METRIC>_member_ens_mean`, `<METRIC>_member_ens_std`, `<METRIC>_member_qpostXX`, `<METRIC>_member_qpreXX`, `<METRIC>_regional_ensmean`, valid-weight and valid-fraction fields, plus `region_area_weight_sum`. `XX` is each configured quantile percentile.


## 3. Project setup


In [ ]:
from pathlib import Path
import sys

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))

from configs.output_paths import workflow_output_dirs
from configs.s2s_config import EXPERIMENT_GROUPS, OBS_VARIABLES, PCC_REGIONS, TCC_REGIONS, LEAD_WINDOWS, WEEKLY_WINDOWS, FIFTEEN_DAY_WINDOWS, MONTH_WINDOWS, WINDOW_SCHEMES, AVAILABLE_PCC_METRICS, AVAILABLE_PCC_MAP_METRICS, AVAILABLE_TCC_METRICS
from util.s2s_experiments import build_experiments
from util.s2s_observations import OBS_REGISTRY, get_obs_file


## 4. Read processed data and plot


In [ ]:
from util.s2s_tcc_skill_plotting import S2SSkillPlotConfig, S2SSkillPlotter

base_dir, figure_dir = workflow_output_dirs("fcst_s2s_tcc", f"{window_scheme}_metrics", output_root=DIAGNOSTIC_OUTPUT_ROOT)
regional_dir=base_dir/"regional"
exp_order=tuple(f"{g}:{e}" for g in groups for e in EXPERIMENT_GROUPS[g]["models"])
plotter=S2SSkillPlotter(S2SSkillPlotConfig(skill_dir=regional_dir, diag_mode=True, file_glob="s2s_regional_tcc_rmse_*.nc", groups_include=groups, obs_include=("ERA5","GPM","NOAA-OLR"), obs_by_var={"PRECT":"GPM","FLUT":"NOAA-OLR"}, vars_include=variables, exp_order=exp_order, regions_include=tuple(r["name"] for r in TCC_REGIONS), out_dir=figure_dir, tag="_".join(groups)))
plotter.plot_quadrant_heatmap_one_region(group=groups, region=region, metric=metric, vmin=vmin, vmax=vmax, nlev=nlev, show=show)
